# De la série temporelle aux indicateurs thermiques d'un bâtiment

**Assessment de stage coEnergy — Jihane**

Ce notebook est le rapport exécutable du projet. Il présente la compréhension physique du problème, l'analyse exploratoire des données, les choix méthodologiques, les modèles entraînés et l'analyse critique de leurs erreurs.

Tout le code réutilisable (chargement, canaux d'entrée, codage des cibles, modèles PyTorch, entraînement, métriques, figures) se trouve dans le package `coenergy_assessment` (dossier `src/`). Le notebook ne fait qu'orchestrer et commenter.

**Plan**

1. Le problème physique
2. Configuration et chargement des données
3. Analyse exploratoire
4. Protocole d'évaluation
5. Modèles
6. Expériences et entraînement
7. Comparaison des modèles sur la validation
8. Modèle retenu : évaluation finale et analyse des erreurs
9. Limites et perspectives

## 1. Le problème physique

### 1.1 D'où viennent les données

Les données suivent la démarche de l'article de coEnergy (Nassif, Guernouti, Stabat et Cohen, *Building-specific excitation optimization and simulation-driven machine learning for rapid in situ identification of the heat loss coefficient*). Un bâtiment est décrit par un modèle thermique détaillé, mis sous forme d'état :

$$C\,\dot T = A\,T + E\,U, \qquad Y = J\,T + G\,U$$

où $T$ regroupe les températures des nœuds (air, couches des parois), $U$ les sollicitations (météo, chauffage) et $C$ les capacités thermiques. Une **population** de variantes de ce bâtiment est ensuite tirée par Monte-Carlo : la géométrie est la même, mais les propriétés thermophysiques (conductivités, capacités, épaisseurs, ponts thermiques) varient autour de leur valeur nominale. Chaque variante est simulée 8 heures avec une excitation de chauffage aléatoire, et ses indicateurs sont calculés directement à partir des matrices du modèle.

Ici le bâtiment est `quentin_house`. Chaque simulation est donc **une variante thermique de la même maison, soumise à un scénario de chauffage différent**, sous la même météo.

### 1.2 Ce qu'il faut prédire

À partir de la température intérieure $T_{int}(t)$ et de la puissance de chauffage $U(t)$, il faut retrouver six indicateurs intrinsèques du bâtiment :

| Famille | Indicateur | Unité | Signification physique |
|---|---|---|---|
| **HLC** | `HLC` | W/K | Coefficient de déperdition global : puissance nécessaire pour maintenir 1 K d'écart avec l'extérieur en régime permanent. C'est l'inverse du gain statique entre le chauffage et la température intérieure. |
| **Hb** | `HLC_ext` | W/K | Part des déperditions vers l'extérieur. |
| **Hb** | `HLC_mit` | W/K | Part des déperditions vers les espaces mitoyens. |
| **CDT** | `tau_b` | h | Décharge après saturation : relaxation libre quand toute la masse du bâti a été portée à un écart uniforme. Elle mesure l'inertie structurelle globale. |
| **CDT** | `tau_n` | h | Persistance de la fraîcheur nocturne : relaxation après 8 h de refroidissement de l'air. Elle dépend de la profondeur de pénétration dans les parois. |
| **CDT** | `tau_inf` | h | Temps de séjour moyen : limite de `tau_n` quand la durée de sollicitation tend vers l'infini. |

Deux contraintes physiques relient ces indicateurs :

$$\text{HLC} = \text{HLC}_{ext} + \text{HLC}_{mit} \qquad\text{et}\qquad \tau_n < \tau_\infty < \tau_b$$

La première est donnée par l'énoncé. La seconde découle des définitions (`tau_inf` est la limite de `tau_n` pour une sollicitation infiniment longue, et `tau_b` correspond à une masse entièrement chargée) ; elle est vérifiée sur les données en section 3.

### 1.3 Pourquoi le problème est difficile

- **Les indicateurs décrivent des régimes que la mesure n'atteint pas.** Le HLC est une grandeur de régime permanent et les constantes de temps valent plusieurs dizaines d'heures, alors que la mesure ne dure que 8 heures. Le réseau doit extrapoler à partir d'un régime fortement transitoire.
- **L'état initial des parois est inconnu.** Deux bâtiments identiques dont les murs ne sont pas à la même température ne donnent pas la même courbe.
- **L'information dépend de l'excitation.** Sans chauffage, la température ne répond qu'à la météo : le lien entre puissance et élévation de température, qui porte l'information sur le HLC, n'est pas observé.

Par rapport à l'article, qui ne prédit que le HLC, ce travail prédit les six indicateurs avec un seul modèle et impose les contraintes physiques par construction.

## 2. Configuration et chargement des données

La cellule suivante détecte l'environnement (local ou Kaggle), localise les données et le package, et fixe les réglages communs.

- `QUICK = True` lance un essai de quelques minutes sur un petit sous-ensemble, pour vérifier que tout s'exécute.
- `RETRAIN = False` recharge les modèles déjà sauvegardés dans `outputs/checkpoints/` au lieu de les réentraîner. Le rapport peut ainsi être relu et réexécuté sans GPU.

In [ ]:
import json
import shutil
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

# ----------------------------------------------------------------------------- réglages
QUICK = False        # True : essai rapide (sous-ensemble, 1 époque)
RETRAIN = False      # True : réentraîner même si un modèle sauvegardé existe
EPOCHS = 20          # époques par modèle
SEED = 42
NOISE_C = 0.25       # bruit de capteur testé (°C) : précision typique des sondes de l'article

# ----------------------------------------------------------------------------- chemins
ON_KAGGLE = Path("/kaggle/input").exists()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path(".").resolve()
SEARCH = Path("/kaggle/input") if ON_KAGGLE else ROOT


def find_parent(pattern: str) -> Path:
    # Dossier contenant le premier fichier correspondant au motif (recherche récursive)
    hits = sorted(SEARCH.rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"{pattern} introuvable sous {SEARCH}")
    return hits[0].parent


DATA = find_parent("T.npy")                                  # T.npy, U.npy, Y.npy, target_names.npy
try:
    import coenergy_assessment as ca
except ModuleNotFoundError:                                  # package non installé (Kaggle)
    src = find_parent("targets.py")
    if src.name != "coenergy_assessment":                    # fichiers déposés à plat : on recrée le dossier
        pkg = ROOT / "_pkg" / "coenergy_assessment"
        pkg.mkdir(parents=True, exist_ok=True)
        for f in src.glob("*.py"):
            shutil.copy(f, pkg / f.name)
        src = pkg
    sys.path.insert(0, str(src.parent))
    import coenergy_assessment as ca
from coenergy_assessment import plots

OUT = ROOT / ("outputs_quick" if QUICK else "outputs")      # l'essai rapide ne touche pas aux vrais résultats
CKPT, RES = OUT / "checkpoints", OUT / "results"
CKPT.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)
if ON_KAGGLE:                                                # reprise : modèles d'une exécution précédente
    for f in SEARCH.rglob("checkpoints/*.pt"):
        if not (CKPT / f.name).exists():
            shutil.copy(f, CKPT / f.name)
    print("Modèles déjà disponibles :", sorted(f.stem for f in CKPT.glob("*.pt")))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
plots.set_style()
pd.options.display.float_format = "{:.3f}".format

print("Appareil :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("Données  :", DATA)
print("Sorties  :", OUT)

Les batchs bruts (`batch_*.pt`) ont été convertis une seule fois avec `ca.prepare_data("Data/raw", "Data/processed")`. La température, arrondie à 0,01 °C, est stockée en centièmes de degré et la puissance en watts, toutes deux en `int16` : le million de simulations occupe 1,9 Go au lieu de 3,8 Go, sans aucune perte.

In [ ]:
data = ca.load_data(DATA, n=20_000 if QUICK else None)
N, L = len(data), data.seq_len
targets, hours = data.targets, data.hours
Y = data.Y

print(f"{N:,} simulations de {L} pas d'une minute ({L / 60:.0f} h)".replace(",", " "))
print("Indicateurs :", targets)

## 3. Analyse exploratoire

### 3.1 Les indicateurs cibles

In [ ]:
stats = pd.DataFrame({
    "unité": [ca.UNITS[t] for t in targets],
    "moyenne": Y.mean(0).numpy(), "écart-type": Y.std(0).numpy(),
    "min": Y.amin(0).numpy(), "médiane": Y.median(0).values.numpy(), "max": Y.amax(0).numpy(),
    "asymétrie": (((Y - Y.mean(0)) / Y.std(0)) ** 3).mean(0).numpy(),
}, index=targets)
stats["CV (%)"] = 100 * stats["écart-type"] / stats["moyenne"]
display(stats.round(2))

fig = plots.plot_target_distributions(Y, targets)
plt.show()

**Lecture.**

- Les trois HLC ont une distribution presque symétrique, en cloche. Les trois constantes de temps sont étalées vers les grandes valeurs (asymétrie positive, moyenne supérieure à la médiane) : c'est attendu pour une grandeur de type $\tau = C / H$, rapport de deux paramètres tirés au hasard. Un passage au logarithme les rend plus symétriques, ce qui motive le codage des cibles de la section 4.
- La dispersion relative (coefficient de variation) est de l'ordre de 13 à 25 % selon l'indicateur. Une erreur relative de quelques pour cent est donc nettement meilleure que la simple prédiction de la moyenne, qui sert de plancher.
- Les ordres de grandeur sont cohérents avec la physique : `tau_n` (une dizaine d'heures) est du même ordre que la durée de la mesure, alors que `tau_inf` et `tau_b` (plusieurs dizaines d'heures) la dépassent largement. On s'attend donc à ce que `tau_n` soit la constante de temps la mieux identifiée.

In [ ]:
c = data.col
somme = (Y[:, c("HLC")] - Y[:, c("HLC_mit")] - Y[:, c("HLC_ext")]).abs().max().item()
ordre = ((Y[:, c("tau_n")] < Y[:, c("tau_inf")]) & (Y[:, c("tau_inf")] < Y[:, c("tau_b")])).float().mean().item()
y_rows = np.ascontiguousarray(Y.numpy())                   # une ligne = 6 indicateurs, vue comme un bloc d'octets
n_unique = len(np.unique(y_rows.view(np.dtype((np.void, y_rows.itemsize * y_rows.shape[1]))).ravel()))

print(f"Écart maximal |HLC - HLC_mit - HLC_ext| : {somme:.1e} W/K (précision float32)")
print(f"Simulations vérifiant tau_n < tau_inf < tau_b : {ordre:.2%}")
print(f"Vecteurs d'indicateurs distincts : {n_unique:,} sur {N:,}".replace(",", " "))

fig = plots.plot_correlation(Y, targets)
plt.show()

**Lecture.**

- Les deux contraintes sont vérifiées sur toutes les simulations : la somme des HLC à la précision numérique près, et l'ordre des constantes de temps sans exception. Elles peuvent donc être imposées au modèle sans perdre de généralité.
- Tous les vecteurs d'indicateurs sont distincts : chaque simulation correspond à une variante différente du bâtiment. Il n'y a pas de doublon entre l'entraînement et le test, donc pas de fuite d'information par ce biais.
- `HLC_mit` et `HLC_ext` sont décorrélés : ils ont été tirés indépendamment. Le HLC global est plus lié à `HLC_mit`, qui varie davantage.
- `tau_b` et `tau_inf` sont presque redondants (corrélation proche de 1), et `tau_n` leur est fortement lié. Les trois constantes de temps partagent la même information sur l'inertie : un modèle **multi-sorties** peut en tirer parti.
- Les HLC et les constantes de temps sont anticorrélés : un bâtiment qui perd plus de chaleur se décharge plus vite, conformément à $\tau \sim C / H$.

### 3.2 Les signaux d'entrée

In [ ]:
P_MAX = float(data.U.max())
u_values = torch.unique(data.U[:20_000])
t0 = data.T[:, 0].float() / 100
t_min, t_max = data.T.amin().item() / 100, data.T.amax().item() / 100

print(f"Valeurs prises par U : {u_values.tolist()} W  ->  signal tout-ou-rien, P_max = {P_MAX:.0f} W")
print(f"Température initiale T(0) : de {t0.min().item():.2f} à {t0.max().item():.2f} °C")
print(f"Température sur l'ensemble des simulations : de {t_min:.2f} à {t_max:.2f} °C")

GROUPS = ca.excitation_groups(data.U, P_MAX)
display(pd.DataFrame({"simulations": {k: int(v.sum()) for k, v in GROUPS.items()},
                      "part (%)": {k: 100 * v.float().mean().item() for k, v in GROUPS.items()}}).round(2))

**Lecture.**

- La puissance ne prend que deux valeurs, 0 et $P_{max}$ = 7000 W. C'est le signal **PRBS** de l'article (*Pseudo Random Binary Sequence*) : une suite de blocs dont l'état est tiré au hasard (allumé avec une probabilité $p_{on}$) et dont la durée est tirée entre $t_{min}$ et $2\,t_{min}$.
- Selon les paramètres tirés, la suite peut ne contenir que des blocs éteints (**sans excitation**), que des blocs allumés (**toujours allumé**) ou un mélange (**avec créneaux**). Ces trois régimes ne portent pas la même information ; ils servent à stratifier le découpage et à analyser les erreurs.
- Toutes les simulations partent de la même température d'air. La température initiale ne distingue donc pas les bâtiments : seule la **dynamique** le fait. L'état initial des parois, lui, est inconnu et aléatoire, comme dans l'article.

In [ ]:
g = torch.Generator().manual_seed(SEED)
desc = ca.describe_excitations(data.U)

# Deux exemples par type de signal : sans chauffe, toujours allumé, un créneau, créneaux multiples
kinds = {
    "sans excitation": GROUPS["sans excitation"],
    "toujours allumé": GROUPS["toujours allumé"],
    "un seul créneau": GROUPS["avec créneaux"] & (desc["n_creneaux"] == 1),
    "créneaux multiples": GROUPS["avec créneaux"] & (desc["n_creneaux"] >= 4),
}
ex_idx, ex_titles = [], []
for name, mask in kinds.items():
    idx = mask.nonzero().squeeze(1)
    if len(idx) == 0:
        continue
    for i in idx[torch.randperm(len(idx), generator=g)[:3]]:
        ex_idx.append(int(i))
        ex_titles.append(f"{name} — HLC {Y[i, c('HLC')]:.0f} W/K, tau_b {Y[i, c('tau_b')]:.0f} h")
ex_idx = torch.tensor(ex_idx)
fig = plots.plot_trajectories(hours, data.temperature(ex_idx), data.power(ex_idx), ex_titles,
                              "Exemples de simulations par type d'excitation")
plt.show()

**Lecture.** Chaque allumage produit une montée rapide de la température de l'air, suivie d'une montée plus lente ; chaque extinction produit la décroissance symétrique. Cette forme en deux temps est la signature d'un système à plusieurs constantes de temps : l'air, de faible capacité, réagit en quelques dizaines de minutes, puis les parois stockent la chaleur sur des heures. L'amplitude de la réponse renseigne sur les déperditions et la courbure sur l'inertie. Sans chauffage, la température ne fait que dériver lentement avec la météo et l'état initial des parois.

### 3.3 Description des excitations

Chaque signal $U(t)$ est résumé par quelques grandeurs physiques (`ca.excitation_descriptors`).

In [ ]:
desc_table = pd.DataFrame({
    name: {"définies": int((~v.isnan()).sum()), "moyenne": v.nanmean().item(),
           "min": v[~v.isnan()].min().item(), "médiane": v.nanmedian().item(),
           "max": v[~v.isnan()].max().item()}
    for name, v in desc.items()}).T
display(desc_table.round(2))

special = {
    "un seul créneau de chauffe": kinds["un seul créneau"],
    "au moins 4 créneaux": kinds["créneaux multiples"],
    "chauffe seulement dans la dernière heure": desc["premier_allumage_h"] >= L / 60 - 1,
    "plus de 100 commutations": desc["n_commutations"] > 100,
}
display(pd.DataFrame({"part (%)": {k: 100 * m.float().mean().item() for k, m in special.items()}}).round(2))

fig = plots.plot_histograms(
    {k: desc[k] for k in ["energie_kWh", "taux_allumage", "n_creneaux", "duree_creneau_h",
                          "premier_allumage_h", "instant_moyen_h"]},
    {"energie_kWh": "Énergie injectée (kWh)", "taux_allumage": "Taux d'allumage",
     "n_creneaux": "Nombre de créneaux de chauffe", "duree_creneau_h": "Durée moyenne d'un créneau (h)",
     "premier_allumage_h": "Premier allumage (h)", "instant_moyen_h": "Barycentre de l'énergie (h)"},
    log=True)
plt.show()

**Lecture.**

- L'énergie injectée va de 0 à 56 kWh (7 kW pendant 8 h). Les excitations couvrent donc toute la plage possible, du bâtiment laissé libre au chauffage permanent.
- La majorité des signaux ne comporte qu'un seul créneau de chauffe : ce sont des PRBS à blocs longs. Les signaux à commutations rapides sont rares. Le modèle voit donc surtout des réponses à un échelon, et peu de réponses à des excitations riches en fréquences.
- Quelques simulations ne chauffent qu'à la toute fin : la réponse observée est alors très courte, et l'on s'attend à des erreurs plus fortes sur ces cas. Ce point est vérifié en section 8.

### 3.4 La réponse libre

In [ ]:
free_idx = GROUPS["sans excitation"].nonzero().squeeze(1)[:20_000]
t_free_all = data.temperature(free_idx)
fig = plots.plot_free_response(hours, t_free_all)
plt.show()

drift = t_free_all[:, -1] - t_free_all[:, 0]
print(f"Dérive sur 8 h sans chauffage : médiane {drift.median().item():+.2f} °C, "
      f"de {drift.min().item():+.2f} à {drift.max().item():+.2f} °C")

**Lecture.** Sans chauffage, toutes les simulations suivent la même tendance, imposée par la météo commune, avec une dispersion bien plus faible que l'élévation due au chauffage. Cette **réponse libre moyenne** peut être retranchée de la température : ce qui reste est, en première approximation, la réponse du bâtiment au seul chauffage. C'est l'idée du canal `T − T_libre` testé en section 6. La dispersion résiduelle vient des différences entre bâtiments et de l'état initial des parois : elle porte une information réelle, mais faible.

### 3.5 Ce que les entrées disent des cibles

Pour mesurer l'information disponible, on se place dans le cas le plus simple : le chauffage permanent. Toutes ces simulations reçoivent exactement la même excitation ; seul le bâtiment change.

In [ ]:
on_idx = GROUPS["toujours allumé"].nonzero().squeeze(1)
t_on = data.temperature(on_idx)
rise_1h, rise_8h = t_on[:, 59] - t_on[:, 0], t_on[:, -1] - t_on[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.2))
plots.plot_density(rise_8h, Y[on_idx, c("HLC")], "Élévation de T après 8 h (°C)", "HLC (W/K)",
                   "HLC", ax=axes[0])
plots.plot_density(rise_8h, Y[on_idx, c("tau_b")], "Élévation de T après 8 h (°C)", "tau_b (h)",
                   "tau_b", ax=axes[1])
plots.plot_density(rise_1h / rise_8h, Y[on_idx, c("tau_n")], "Part de l'élévation atteinte en 1 h",
                   "tau_n (h)", "tau_n", ax=axes[2])
fig.suptitle("Chauffage permanent : une même excitation, des bâtiments différents", fontweight="bold")
fig.tight_layout()
plt.show()

feats = {"élévation à 1 h": rise_1h, "élévation à 8 h": rise_8h, "rapport 1 h / 8 h": rise_1h / rise_8h}
corr_on = pd.DataFrame({name: [torch.corrcoef(torch.stack([f, Y[on_idx, j]]))[0, 1].item()
                               for j in range(6)] for name, f in feats.items()}, index=targets)
print("Chauffage permanent — corrélation entre un résumé de T(t) et chaque indicateur :")
display(corr_on.round(2))

drift_all = data.temperature(GROUPS["sans excitation"].nonzero().squeeze(1))
drift_all = drift_all[:, -1] - drift_all[:, 0]
y_free = Y[GROUPS["sans excitation"]]
print("Sans excitation — corrélation entre la dérive de T sur 8 h et chaque indicateur :")
display(pd.DataFrame({"dérive sur 8 h": [torch.corrcoef(torch.stack([drift_all, y_free[:, j]]))[0, 1].item()
                                          for j in range(6)]}, index=targets).round(2))

**Lecture.** À excitation identique, l'élévation de température est liée aux indicateurs : plus le bâtiment perd de chaleur, moins il monte en température. Mais la relation est un nuage épais, pas une courbe : un seul résumé de la série ne suffit pas, et l'inertie et l'état initial des parois brouillent le lien. C'est ce qui justifie un modèle séquentiel, qui exploite toute la forme de la courbe. Sans excitation, le lien entre la dérive de température et les indicateurs est plus faible : on attend de moins bonnes performances sur ce régime.

Sur les simulations à chauffage permanent, l'élévation de température après 8 h est le résumé le plus lié au HLC (corrélation de −0,59) et à `HLC_mit` (−0,52) : à puissance égale, un bâtiment qui perd davantage monte moins haut. `tau_n` est surtout lié à l'élévation de la **première heure** (−0,35), ce qui est cohérent avec sa définition : elle décrit la réponse de l'air et des couches superficielles des parois. En revanche, `tau_b` et `tau_inf` ne sont presque pas corrélés à ces résumés (|r| ≤ 0,10) : les constantes de temps longues ne laissent qu'une trace faible dans 8 heures de mesure. On s'attend donc à ce qu'elles soient les indicateurs les plus difficiles à prédire.

### 3.6 Ce que l'analyse exploratoire implique pour la modélisation

| Constat | Conséquence |
|---|---|
| Deux contraintes physiques exactes dans les données | Codage des cibles qui les respecte par construction |
| Constantes de temps asymétriques et fortement corrélées | Passage au logarithme ; un seul modèle multi-sorties |
| Trois régimes d'excitation d'information très inégale | Découpage stratifié ; métriques calculées par régime |
| Même météo et même T(0) pour toutes les simulations | Normalisation globale de T ; canal `T − T_libre` |
| L'amplitude de la réponse porte l'information sur le HLC | Pas de normalisation par simulation, qui effacerait cette amplitude |
| L'information dépend de l'excitation | Analyse des erreurs en fonction des descripteurs de l'excitation |

## 4. Protocole d'évaluation

### 4.1 Découpage

Les simulations sont réparties en entraînement (80 %), validation (10 %) et test (10 %). Le tirage est **stratifié par régime d'excitation**, pour que le régime rare (chauffage permanent, moins de 1 %) soit représenté dans les trois ensembles. Le découpage est enregistré et partagé par toutes les expériences.

- L'**entraînement** sert à ajuster les poids et toutes les statistiques de normalisation.
- La **validation** sert à choisir l'époque, à comparer les modèles et à retenir le meilleur.
- Le **test** n'est utilisé qu'une seule fois, à la fin, pour le modèle retenu.

In [ ]:
SPLITS = ca.make_splits(GROUPS, None if QUICK else DATA / "splits.pt" if (DATA / "splits.pt").exists()
                        else OUT / "splits.pt", seed=SEED)
display(pd.DataFrame({split: {"simulations": len(idx),
                              **{f"{k} (%)": 100 * m[idx].float().mean().item() for k, m in GROUPS.items()}}
                      for split, idx in SPLITS.items()}).T.round(2))

assert len(set(SPLITS["train"].tolist()) & set(SPLITS["test"].tolist())) == 0
assert len(set(SPLITS["val"].tolist()) & set(SPLITS["test"].tolist())) == 0
print("Aucune simulation commune entre les ensembles.")

### 4.2 Entrées du réseau

La température est normalisée avec **une seule** moyenne et un seul écart-type, calculés sur l'entraînement, et la puissance est divisée par $P_{max}$. Une normalisation par simulation effacerait l'amplitude de la réponse, qui porte l'information sur le HLC.

Deux jeux de canaux sont comparés (`ca.InputScaler`) :

| Jeu | Canaux |
|---|---|
| `base` (2) | température normalisée, $U / P_{max}$ |
| `physique` (5) | les deux précédents, `T − T_libre` (réponse au chauffage, approchée), `T − T(0)` (élévation depuis le départ) et le temps normalisé (position dans la météo commune) |

In [ ]:
scaler = ca.InputScaler.fit(data, SPLITS["train"], GROUPS["sans excitation"], P_MAX)
print(f"T : moyenne {scaler.t_mu:.2f} °C, écart-type {scaler.t_sd:.2f} °C")
print(f"Écart-type de T - T_libre : {scaler.d_free_sd:.2f} °C | de T - T(0) : {scaler.d_t0_sd:.2f} °C")

x = scaler.build(data.temperature(ex_idx[-1:]), data.power(ex_idx[-1:]), "physique")[0]
fig, ax = plt.subplots(figsize=(9.5, 3.6))
for k, name in enumerate(["T normalisée", "U / P_max", "T − T_libre", "T − T(0)", "temps"]):
    ax.plot(hours, x[k].numpy(), label=name)
ax.set(xlabel="Temps (h)", ylabel="Valeur normalisée", title="Les 5 canaux d'entrée pour une simulation")
ax.legend(ncols=5, loc="upper left")
plt.show()

### 4.3 Codage des cibles

Le réseau produit 5 sorties, jamais 6 : le HLC global n'est pas prédit, il est **reconstruit** comme la somme de ses deux composantes. La contrainte $\text{HLC} = \text{HLC}_{ext} + \text{HLC}_{mit}$ est donc exacte quel que soit le modèle. Deux codages sont comparés pour les constantes de temps (`ca.TargetCodec`) :

| Codage | Sorties du réseau | Ordre des constantes de temps |
|---|---|---|
| `contraint` | `HLC_mit`, `HLC_ext`, $\log \tau_n$, $\log(\tau_\infty - \tau_n)$, $\log(\tau_b - \tau_\infty)$ | garanti : chaque écart est une exponentielle, donc positif |
| `direct` | `HLC_mit`, `HLC_ext`, $\tau_b$, $\tau_n$, $\tau_\infty$ | non garanti, vérifié après coup |

Les sorties sont ensuite standardisées (moyenne et écart-type de l'entraînement), pour que les cinq termes pèsent autant dans la perte.

In [ ]:
for encoding in ("contraint", "direct"):
    codec = ca.TargetCodec(encoding, targets).fit(Y[SPLITS["train"]])
    back = codec.inverse(codec.transform(Y[:2000]))
    print(f"Codage {encoding:9s} : erreur maximale d'un aller-retour = {(back - Y[:2000]).abs().max().item():.1e}")

### 4.4 Métriques

Toutes les métriques sont calculées **en unités physiques**, sur les indicateurs décodés, et non sur les sorties standardisées du réseau :

- **RMSE** et **MAE** : erreur quadratique et absolue moyennes ;
- **P95** : 95ᵉ centile de l'erreur absolue. C'est le critère de fiabilité de l'article : l'erreur reste sous ce seuil pour 95 % des bâtiments ;
- **MAPE** : erreur relative moyenne ;
- **R²** : part de la variance expliquée.

Elles sont données par indicateur, puis regroupées par **famille** (HLC, Hb, CDT) et par **régime d'excitation**. Pour classer les modèles avec un seul nombre, on utilise la RMSE normalisée moyenne $\frac{1}{6}\sum \sqrt{1 - R^2}$ : elle vaut 1 pour la prédiction de la moyenne et 0 pour une prédiction parfaite.

### 4.5 Modèles de référence

Deux références simples fixent l'échelle :

- la **moyenne** de l'entraînement, plancher absolu ;
- une **régression linéaire régularisée** sur un résumé de la simulation (élévation de température et énergie cumulée toutes les 30 minutes). Elle indique ce qu'un modèle linéaire tire des données : c'est le niveau qu'un réseau de neurones doit dépasser pour se justifier.

In [ ]:
Y_VAL = Y[SPLITS["val"]]
GROUPS_VAL = {k: m[SPLITS["val"]] for k, m in GROUPS.items()}
VAL_PRED: dict[str, torch.Tensor] = {}       # prédictions de chaque modèle sur la validation

VAL_PRED["Moyenne"] = ca.mean_baseline(Y[SPLITS["train"]], len(Y_VAL))

ridge_idx = SPLITS["train"][:200_000]
ridge = ca.RidgeBaseline().fit(ca.summary_features(data, ridge_idx), Y[ridge_idx])
VAL_PRED["Régression linéaire"] = ridge.predict(ca.summary_features(data, SPLITS["val"]))

for name in ("Moyenne", "Régression linéaire"):
    print(f"\n{name} (validation)")
    display(ca.regression_metrics(Y_VAL, VAL_PRED[name], targets).round(3))

## 5. Modèles

Deux architectures séquentielles sont comparées (`coenergy_assessment/models.py`). Elles ont la même interface, une séquence de 480 pas en entrée et 5 sorties, et se terminent par la même tête de régression.

| Modèle | Principe | Pourquoi l'essayer |
|---|---|---|
| **CNN-GRU** | 3 blocs convolutifs (motifs locaux, longueur divisée par 8), puis un GRU bidirectionnel à 2 couches | C'est l'architecture de l'article de référence : la convolution capte les fronts de montée, le GRU la dynamique lente |
| **Transformer à patches** | 30 patches de 16 minutes, un token `[CLS]`, position apprise, 4 couches d'attention | L'attention relie directement une variation de température au créneau de chauffe qui l'a causée, même éloigné |

Aucun modèle n'est causal : la séquence entière est observée avant la prédiction, ce qui correspond à l'usage réel (on identifie le bâtiment une fois la mesure terminée).

In [ ]:
rows = {}
for name in ("cnn_gru", "transformer"):
    for inputs, n_ch in ca.N_CHANNELS.items():
        model = ca.build_model(name, n_ch, seq_len=L)
        with torch.no_grad():
            out = model(torch.zeros(2, n_ch, L))
        rows[(name, inputs)] = {"canaux": n_ch, "paramètres": ca.count_parameters(model), "sortie": tuple(out.shape)}
display(pd.DataFrame(rows).T)
del model

## 6. Expériences et entraînement

Chaque expérience ne change qu'**un seul** élément par rapport au CNN-GRU de référence, pour que les écarts soient attribuables :

| Question | Expériences comparées |
|---|---|
| Le codage contraint coûte-t-il de la précision ? | CNN-GRU `contraint` contre CNN-GRU `direct` |
| Les canaux physiques aident-ils ? | chaque architecture avec les entrées `base` et `physique` |
| Quelle architecture est la meilleure ? | CNN-GRU et Transformer, à entrées et codage identiques |

**Réglages communs** (`ca.Experiment`) : optimiseur AdamW, planification *one-cycle* du taux d'apprentissage (montée sur 10 % des pas, puis décroissance en cosinus), lots de 512, écrêtage du gradient à 1, précision mixte sur GPU, perte quadratique sur les sorties standardisées, même graine. Le modèle conservé est celui de la meilleure époque en validation.


Les séries tiennent entièrement sur le GPU sous leur forme compacte : les lots sont formés par indexation directe, sans `DataLoader`, ce qui accélère fortement l'entraînement.

In [ ]:
common = dict(epochs=1 if QUICK else EPOCHS, seed=SEED, n_train=1_000 if QUICK else None)

EXPERIMENTS = {
    "CNN-GRU": ca.Experiment("cnn_gru_base_contraint", "cnn_gru", "base", "contraint", **common),
    "CNN-GRU, codage direct": ca.Experiment("cnn_gru_base_direct", "cnn_gru", "base", "direct", **common),
    "CNN-GRU, canaux physiques": ca.Experiment("cnn_gru_physique_contraint", "cnn_gru", "physique", "contraint", **common),
    "Transformer": ca.Experiment("transformer_base_contraint", "transformer", "base", "contraint", **common),
    "Transformer, canaux physiques": ca.Experiment("transformer_physique_contraint", "transformer", "physique", "contraint", **common),
}
display(pd.DataFrame({label: {"modèle": e.model, "entrées": e.inputs, "codage": e.encoding,
                              "époques": e.epochs, "lot": e.batch_size, "lr max": e.lr}
                      for label, e in EXPERIMENTS.items()}).T)

In [ ]:
data_dev, scaler_dev = data.to(DEVICE), scaler.to(DEVICE)
RUNS: dict[str, ca.RunResult] = {}

for label, exp in EXPERIMENTS.items():
    RUNS[label] = ca.run_experiment(exp, data_dev, SPLITS, scaler_dev, CKPT, retrain=RETRAIN)
    run = RUNS[label]
    VAL_PRED[label] = ca.predict(run.model, data_dev, SPLITS["val"], scaler_dev, run.codec, exp.inputs)
    score = ca.global_score(ca.regression_metrics(Y_VAL, VAL_PRED[label], targets))
    print(f"{label:32s} meilleure époque {run.best_epoch:2d} | {run.train_seconds / 60:5.1f} min | "
          f"RMSE normalisée moyenne (validation) = {score:.4f}")

In [ ]:
fig = plots.plot_learning_curves({label: run.history for label, run in RUNS.items()})
plt.show()

**Lecture.**

- Les cinq modèles atteignent leur meilleure validation à la dernière ou à l'avant-dernière époque : les courbes baissent encore à la fin de l'entraînement. Plus d'époques amélioreraient probablement un peu les scores.
- L'écart entre entraînement et validation est très faible (0,716 contre 0,722 pour le CNN-GRU à canaux physiques) : il n'y a **pas de surapprentissage**. Les modèles sont plutôt limités par l'information disponible et par leur capacité.
- Sur un GPU Kaggle, un CNN-GRU demande environ 11 minutes pour 20 époques sur 800 000 simulations, un Transformer environ 9 minutes.

Les pertes de l'expérience à codage direct portent sur des cibles différentes : elles ne se comparent pas en valeur aux autres. La comparaison se fait en unités physiques, dans la section suivante.

## 7. Comparaison des modèles sur la validation

### 7.1 Vue d'ensemble

In [ ]:
VAL_METRICS = {label: ca.regression_metrics(Y_VAL, pred, targets) for label, pred in VAL_PRED.items()}

summary = pd.DataFrame({
    label: {"score": ca.global_score(m),
            **{f"R² {fam}": v for fam, v in ca.family_summary(m)["R2"].items()},
            "paramètres": RUNS[label].n_parameters if label in RUNS else np.nan,
            "minutes": RUNS[label].train_seconds / 60 if label in RUNS else np.nan}
    for label, m in VAL_METRICS.items()}).T.sort_values("score")
print("Validation — score = RMSE normalisée moyenne (plus petit = meilleur)")
display(summary.round(4))

r2 = pd.DataFrame({label: m["R2"] for label, m in VAL_METRICS.items()}).T
rmse = pd.DataFrame({label: m["RMSE"] for label, m in VAL_METRICS.items()}).T
p95 = pd.DataFrame({label: m["P95"] for label, m in VAL_METRICS.items()}).T
print("RMSE par indicateur (W/K pour les HLC, h pour les constantes de temps)")
display(rmse.round(3))
print("P95 de l'erreur absolue par indicateur")
display(p95.round(3))

fig = plots.plot_model_comparison(rmse.loc[summary.index], "RMSE", "RMSE par indicateur — validation")
plt.show()

**Lecture.**

- **Les cinq réseaux battent nettement les deux références.** Le meilleur, le CNN-GRU à canaux physiques, obtient un score de 0,845, contre 0,947 pour la régression linéaire et 1 pour la moyenne.
- **Les trois variantes du CNN-GRU sont très proches** (0,845 à 0,848). Les deux Transformer sont en retrait (0,879 et 0,884), sur tous les indicateurs : l'architecture de l'article de référence est la mieux adaptée ici.
- **Le classement des indicateurs suit l'analyse exploratoire.** `HLC_ext` (R² 0,48) et le HLC global (0,44) sont les mieux prédits, `HLC_mit` l'est moins (0,28). Parmi les constantes de temps, `tau_n`, du même ordre que la durée de mesure, est la mieux identifiée (0,26) ; `tau_b` (0,13) et `tau_inf` (0,10), plusieurs fois plus longues que la mesure, le sont très mal.
- **En valeur relative**, l'erreur moyenne est de 8,3 % sur le HLC, 9,1 % sur `HLC_ext`, 15,8 % sur `HLC_mit` et de 12 à 18 % sur les constantes de temps.
- **La précision reste modeste en absolu** : aucun indicateur ne dépasse un R² de 0,5. Une grande partie de la variabilité des bâtiments n'est pas récupérable à partir de 8 heures de mesure, ou pas avec ces modèles.

### 7.2 Erreurs par famille d'indicateurs

In [ ]:
families = pd.concat({label: ca.family_summary(m)[["RMSE", "P95", "MAPE", "R2"]]
                      for label, m in VAL_METRICS.items()}, names=["modèle"])
display(families.unstack("famille").round(3))

### 7.3 Le codage contraint coûte-t-il de la précision ?

In [ ]:
pair = ["CNN-GRU", "CNN-GRU, codage direct"]
display(pd.concat({label: VAL_METRICS[label][["RMSE", "P95", "R2"]] for label in pair}, axis=1).round(3))

viol = pd.DataFrame({label: ca.constraint_violations(VAL_PRED[label], targets) for label in VAL_PRED}).T * 100
print("Prédictions violant une contrainte physique (%)")
display(viol.round(3))

**Lecture.** Les deux codages donnent des précisions équivalentes : les RMSE diffèrent de moins de 1 % sur chaque indicateur (23,74 contre 23,82 W/K sur le HLC, 9,51 contre 9,46 h sur `tau_b`), et les scores globaux sont identiques (0,848). Le codage direct ne viole pas non plus l'ordre des constantes de temps sur la validation : les écarts moyens entre `tau_n`, `tau_inf` et `tau_b` (environ 20 h chacun) sont grands devant les erreurs, et le modèle reste proche de la moyenne. Le codage contraint reste préférable : il apporte une **garantie** par construction, sans coût en précision, ce qui compte dès que le modèle est appliqué à des cas éloignés de ceux de l'entraînement.

### 7.4 Les canaux physiques aident-ils ?

In [ ]:
gain = pd.DataFrame({
    arch: 100 * (rmse.loc[f"{arch}, canaux physiques"] / rmse.loc[arch] - 1)
    for arch in ("CNN-GRU", "Transformer")}).T
print("Variation de la RMSE avec les canaux physiques (%, négatif = amélioration)")
display(gain.round(1))

**Lecture.** Le gain est faible mais systématique : les canaux physiques réduisent la RMSE de 0,2 à 1,1 % sur tous les indicateurs et pour les deux architectures, le plus nettement sur `HLC_ext` (−1,0 % pour le CNN-GRU, −1,1 % pour le Transformer). Le réseau retrouve donc presque seul l'information qu'on lui fournit explicitement. Les canaux physiques accélèrent surtout l'apprentissage : à la 10ᵉ époque, la perte de validation du CNN-GRU est de 0,739 avec eux contre 0,748 sans.

### 7.5 Performances par régime d'excitation

In [ ]:
BEST = summary.index[[label in RUNS for label in summary.index]][0]      # meilleur réseau en validation
print("Modèle retenu :", BEST)

by_group = ca.metrics_by_group(Y_VAL, VAL_PRED[BEST], targets, GROUPS_VAL)
display(by_group["R2"].unstack("régime").round(3))
fig = plots.plot_group_bars(by_group["R2"].unstack("régime")[list(GROUPS_VAL)].loc[targets],
                            "R² (validation)", f"{BEST} — R² par régime d'excitation")
plt.show()

**Lecture.**

- **Sans excitation, le HLC global et `HLC_mit` se dégradent nettement** (R² 0,28 contre 0,45 avec créneaux, et 0,17 contre 0,28) : sans chauffage, la réponse du bâtiment à une puissance connue n'est pas observée.
- **`HLC_ext` est, lui, aussi bien prédit sans excitation qu'avec** (R² 0,49 contre 0,48). Interprétation : la météo joue le rôle d'excitation pour les déperditions vers l'extérieur. La dérive libre de la température, imposée par l'écart avec l'extérieur, dépend directement de `HLC_ext`, alors que les espaces mitoyens, plus stables, ne sollicitent pas le bâtiment de la même façon.
- **Les constantes de temps longues sont même un peu mieux prédites sans excitation** (`tau_b` : 0,19 contre 0,12), probablement parce que la relaxation libre depuis l'état initial des parois renseigne directement sur l'inertie.
- **Le chauffage permanent donne les meilleurs résultats** pour les HLC et `tau_n`, mais ce régime ne compte que 871 simulations de validation : ses métriques sont moins précises.

## 8. Modèle retenu : évaluation finale et analyse des erreurs

Le modèle retenu est le réseau qui obtient le meilleur score en validation. Le test n'a servi à aucune décision jusqu'ici : il est utilisé une seule fois, ci-dessous.

### 8.1 Résultats sur le test

In [ ]:
best_run = RUNS[BEST]
Y_TEST = Y[SPLITS["test"]]
GROUPS_TEST = {k: m[SPLITS["test"]] for k, m in GROUPS.items()}
TEST_PRED = ca.predict(best_run.model, data_dev, SPLITS["test"], scaler_dev, best_run.codec,
                       best_run.experiment.inputs)

test_metrics = ca.regression_metrics(Y_TEST, TEST_PRED, targets)
test_metrics.insert(0, "unité", [ca.UNITS[t] for t in targets])
print(f"{BEST} — test ({len(Y_TEST)} simulations)")
display(test_metrics.round(3))
print("Par famille")
display(ca.family_summary(test_metrics.drop(columns="unité")).round(3))
print("Contraintes physiques violées (%) :",
      {k: round(100 * v, 3) for k, v in ca.constraint_violations(TEST_PRED, targets).items()})

compare = pd.DataFrame({"validation": VAL_METRICS[BEST]["RMSE"], "test": test_metrics["RMSE"]})
compare["écart (%)"] = 100 * (compare["test"] / compare["validation"] - 1)
print("RMSE : validation contre test")
display(compare.round(3))

**Lecture.**

- **Les métriques du test sont pratiquement identiques à celles de la validation** (écarts inférieurs à 1 %) : le choix du modèle sur la validation n'a pas surajusté.
- **HLC** : RMSE de 23,5 W/K (8,2 % en moyenne) ; pour 95 % des bâtiments, l'erreur reste sous 46 W/K, soit environ 20 % de la valeur moyenne.
- **Hb** : 12,3 W/K sur `HLC_ext` (9,0 %), 22,3 W/K sur `HLC_mit` (15,7 %).
- **CDT** : 1,7 h sur `tau_n` (12,2 %), 9,4 h sur `tau_b` (14,7 %), 7,2 h sur `tau_inf` (17,8 %).
- **Aucune prédiction ne viole une contrainte physique.**

### 8.2 Prédiction contre référence

In [ ]:
fig = plots.plot_parity(Y_TEST, TEST_PRED, targets, f"{BEST} — prédiction contre référence (test)")
plt.show()

err = TEST_PRED - Y_TEST
bias = pd.DataFrame({"biais moyen": err.mean(0).numpy(),
                     "biais, tiers inférieur": [err[Y_TEST[:, j] <= Y_TEST[:, j].quantile(1 / 3), j].mean().item() for j in range(6)],
                     "biais, tiers supérieur": [err[Y_TEST[:, j] >= Y_TEST[:, j].quantile(2 / 3), j].mean().item() for j in range(6)]},
                    index=targets)
display(bias.round(3))

**Lecture.** Les nuages suivent la diagonale mais sont **aplatis** : l'écart-type des prédictions ne vaut que 66 % de celui des valeurs vraies pour le HLC, 52 % pour `HLC_mit`, 37 % pour `tau_b` et 32 % pour `tau_inf`. Le biais est positif dans le tiers inférieur (+19 W/K sur le HLC) et négatif dans le tiers supérieur (−20 W/K) : c'est une **régression vers la moyenne** marquée, d'autant plus forte que l'indicateur est mal identifiable. Le biais moyen reste presque nul (+0,2 W/K sur le HLC), sauf pour `tau_b` (−1,1 h) et `tau_inf` (−0,9 h), légèrement sous-estimés à cause de leur distribution étalée vers les grandes valeurs.

### 8.3 Erreur par régime d'excitation

In [ ]:
test_by_group = ca.metrics_by_group(Y_TEST, TEST_PRED, targets, GROUPS_TEST)
display(test_by_group[["RMSE", "P95", "R2"]].unstack("régime").round(3))

### 8.4 Erreur en fonction de l'excitation

L'article optimise l'excitation en reliant l'erreur de prédiction aux paramètres du signal. La même lecture est faite ici : les simulations avec chauffage du test sont regroupées en tranches de même effectif selon un descripteur de l'excitation, et l'erreur est calculée dans chaque tranche.

In [ ]:
heated = ~GROUPS_TEST["sans excitation"]
desc_test = {k: v[SPLITS["test"]] for k, v in desc.items()}
for key, xlabel in [("energie_kWh", "Énergie injectée (kWh)"),
                    ("premier_allumage_h", "Instant du premier allumage (h)"),
                    ("n_commutations", "Nombre de commutations")]:
    table = plots.binned_errors(desc_test[key][heated], Y_TEST[heated], TEST_PRED[heated], targets)
    table.to_csv(RES / f"erreur_selon_{key}.csv", index=False)
    fig = plots.plot_binned_errors(table, targets, xlabel, f"Erreur selon : {xlabel[0].lower() + xlabel[1:]}")
    plt.show()

**Lecture.** Résultat important : **l'erreur dépend peu de l'excitation.**

- Avec l'énergie injectée, la RMSE du HLC passe de 24,2 W/K (6 kWh) à 22,5 W/K (46 kWh), soit seulement −7 % ; les autres indicateurs varient de moins de 5 %.
- Selon l'instant du premier allumage, l'erreur sur le HLC est la plus faible pour un démarrage vers 2 h (22,4 W/K) et remonte pour un démarrage après 6 h (23,9 W/K). `HLC_ext` s'améliore au contraire quand le chauffage démarre tard (13,2 puis 11,7 W/K), ce qui rejoint la section 7.5 : une longue phase libre renseigne sur les déperditions vers l'extérieur.
- Les signaux à commutations nombreuses dégradent légèrement `HLC_ext` (11,6 puis 13,8 W/K) : des créneaux courts chargent peu les parois.

Dans ce jeu de données, l'excitation n'est donc pas le facteur limitant principal. La limite vient plutôt de l'état initial inconnu des parois et de la durée de mesure, courte devant les constantes de temps. Cela nuance, pour ce cas, la conclusion de l'article : celui-ci optimise l'excitation pour une durée et une météo connues et pour le seul HLC, avec un modèle réentraîné pour chaque essai.

### 8.5 Erreur en fonction de la valeur de l'indicateur

In [ ]:
rows = []
for j, name in enumerate(targets):
    table = plots.binned_errors(Y_TEST[:, j], Y_TEST, TEST_PRED, targets, n_bins=10)
    rows.append(table[table["indicateur"] == name])
by_value = pd.concat(rows)
fig, axes = plt.subplots(2, 3, figsize=(13.5, 6.4))
for ax, name in zip(axes.flat, targets):
    sub = by_value[by_value["indicateur"] == name]
    ax.plot(sub["centre"], sub["RMSE"], marker="o", ms=4, label="RMSE")
    ax.plot(sub["centre"], sub["P95"], marker="o", ms=4, label="P95 de l'erreur absolue")
    ax.set(title=name, xlabel=f"Valeur de référence ({ca.UNITS[name]})")
    ax.set_ylim(bottom=0)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper right", ncols=2)
fig.suptitle("Erreur selon la valeur de l'indicateur (test)", fontweight="bold", x=0.02, ha="left")
fig.tight_layout()
plt.show()

**Lecture.** L'erreur est minimale au centre de la distribution et augmente aux deux extrêmes, conséquence directe de la régression vers la moyenne. Pour les constantes de temps, elle est nettement plus forte du côté des grandes valeurs : la RMSE de `tau_b` atteint 20,0 h sur les 10 % de bâtiments les plus inertes, contre 11,9 h sur les 10 % les moins inertes (15,7 contre 8,8 h pour `tau_inf`). Comme attendu, plus le bâtiment est inerte, plus 8 heures de mesure sont courtes devant sa dynamique, et moins il est bien caractérisé.

### 8.6 Robustesse au bruit de mesure

Les simulations sont sans bruit, alors qu'une sonde réelle en a. L'article entraîne ses modèles avec un bruit gaussien calibré sur les capteurs (précision typique de ±0,25 °C). Deux questions sont posées :

1. que devient le modèle retenu si la température est bruitée au moment de la prédiction ?
2. un modèle entraîné avec du bruit résiste-t-il mieux ?

In [ ]:
noisy_exp = ca.Experiment(**{**asdict(best_run.experiment), "name": best_run.experiment.name + "_bruit",
                             "noise_c": NOISE_C})
noisy_run = ca.run_experiment(noisy_exp, data_dev, SPLITS, scaler_dev, CKPT, retrain=RETRAIN)

rows = {}
for sigma in (0.0, 0.05, 0.1, 0.25, 0.5):
    for label, run in [("entraîné sans bruit", best_run), (f"entraîné avec {NOISE_C} °C de bruit", noisy_run)]:
        pred = ca.predict(run.model, data_dev, SPLITS["val"], scaler_dev, run.codec,
                          run.experiment.inputs, noise_c=sigma, seed=SEED)
        m = ca.regression_metrics(Y_VAL, pred, targets)
        rows[(label, sigma)] = {"score": ca.global_score(m), **m["RMSE"].to_dict()}
noise_table = pd.DataFrame(rows).T.rename_axis(["modèle", "bruit à la prédiction (°C)"])
noise_table.to_csv(RES / "robustesse_bruit.csv")
display(noise_table.round(3))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for label in noise_table.index.get_level_values(0).unique():
    sub = noise_table.loc[label]
    ax.plot(sub.index, sub["score"], marker="o", ms=5, label=label)
ax.set(xlabel="Bruit ajouté à T à la prédiction (°C)", ylabel="RMSE normalisée moyenne (validation)",
       title="Sensibilité au bruit de mesure")
ax.set_ylim(bottom=0)
ax.legend()
plt.show()

**Lecture.** C'est le résultat le plus important pour un usage réel.

- **Le modèle entraîné sans bruit est très fragile.** 0,05 °C de bruit suffit à faire passer son score de 0,845 à 0,974. À 0,25 °C, la précision typique d'une sonde, il ne fait pas mieux que la prédiction de la moyenne (0,997). Il exploite des détails de la courbe bien plus fins que ce qu'une mesure réelle peut fournir (les simulations sont arrondies à 0,01 °C).
- **Le modèle entraîné avec 0,25 °C de bruit est robuste.** Il perd en précision sur des données propres (0,909 contre 0,845), mais reste stable quand le bruit augmente : 0,917 à 0,25 °C et 0,932 à 0,5 °C.

Pour des mesures réelles, c'est donc le modèle entraîné avec bruit qu'il faut retenir, comme le fait l'article. Le score réaliste du projet est d'environ 0,92, et non 0,845.

### 8.7 Sauvegarde des résultats

Tous les tableaux sont enregistrés dans `outputs/results/`, et les modèles dans `outputs/checkpoints/`.

In [ ]:
results = {
    "config": {"quick": QUICK, "epochs": EPOCHS, "seed": SEED, "n_simulations": N, "device": str(DEVICE),
               "splits": {k: len(v) for k, v in SPLITS.items()}},
    "experiments": {label: {**asdict(run.experiment), "n_parameters": run.n_parameters,
                            "best_epoch": run.best_epoch, "train_minutes": run.train_seconds / 60,
                            "history": run.history} for label, run in RUNS.items()},
    "validation": {label: m.to_dict(orient="index") for label, m in VAL_METRICS.items()},
    "validation_scores": summary["score"].to_dict(),
    "validation_violations_pct": viol.to_dict(orient="index"),
    "best_model": BEST,
    "validation_by_group": {str(k): v for k, v in by_group.to_dict(orient="index").items()},
    "test": test_metrics.drop(columns="unité").to_dict(orient="index"),
    "test_by_group": {str(k): v for k, v in test_by_group.to_dict(orient="index").items()},
    "test_bias": bias.to_dict(orient="index"),
    "noise": {str(k): v for k, v in noise_table.to_dict(orient="index").items()},
    "eda": {"targets": stats.drop(columns="unité").to_dict(orient="index"),
            "corr_always_on": corr_on.to_dict(orient="index"),
            "groups_pct": {k: 100 * v.float().mean().item() for k, v in GROUPS.items()}},
}
(RES / "results.json").write_text(json.dumps(results, indent=1, ensure_ascii=False), encoding="utf-8")
summary.to_csv(RES / "validation_resume.csv")
rmse.to_csv(RES / "validation_rmse.csv")
test_metrics.to_csv(RES / "test_metriques.csv")
np.savez_compressed(RES / "test_predictions.npz", y_true=Y_TEST.numpy(), y_pred=TEST_PRED.numpy(),
                    idx=SPLITS["test"].numpy())
print("Fichiers écrits dans", RES, ":", sorted(p.name for p in RES.iterdir()))

Sur Kaggle, le panneau « Output » ne permet pas toujours de télécharger un dossier. La cellule suivante regroupe tout `outputs/` dans une seule archive, `outputs.zip`, avec un lien de téléchargement.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive(str(ROOT / "outputs"), "zip", root_dir=ROOT, base_dir="outputs")
print(f"Archive : {archive} ({Path(archive).stat().st_size / 1e6:.1f} Mo)")
display(FileLink(Path(archive).name) if ON_KAGGLE else FileLink(archive))

## 9. Limites et perspectives

**Synthèse.** Le modèle retenu est le CNN-GRU à 5 canaux et codage contraint. Sur le test, il prédit le HLC avec une RMSE de 23,5 W/K (8,2 %, P95 de 46 W/K), `HLC_ext` à 12,3 W/K (9,0 %), `HLC_mit` à 22,3 W/K (15,7 %), et les constantes de temps à 1,7 h (`tau_n`), 9,4 h (`tau_b`) et 7,2 h (`tau_inf`), en respectant toujours les deux contraintes physiques. Il bat la régression linéaire sur tous les indicateurs, mais explique moins de la moitié de la variance : les constantes de temps longues restent peu identifiables en 8 heures. Le codage contraint ne coûte rien en précision, les canaux physiques apportent un gain faible et le CNN-GRU devance le Transformer. Enfin, la robustesse au bruit est le point critique : seul un modèle entraîné avec du bruit reste utilisable sur des mesures réelles, avec un score d'environ 0,92.

**Limites**

- **Un seul bâtiment et une seule météo.** Toutes les simulations sont des variantes de `quentin_house` sous les mêmes conditions extérieures. Le modèle a appris cette configuration : il ne se transpose pas tel quel à un autre bâtiment ou à un autre jour. C'est cohérent avec la démarche de l'article, qui réentraîne un modèle pour chaque bâtiment et chaque essai.
- **Des données simulées.** Le modèle hérite des hypothèses du simulateur (zones isothermes, transferts unidimensionnels, absence d'infiltrations et d'occupants). L'écart avec une mesure réelle n'est pas évalué ici ; seule la robustesse au bruit de capteur l'est.
- **Identifiabilité.** Huit heures de mesure sont courtes devant les grandes constantes de temps, et une partie des simulations n'a aucune excitation. Sur ces cas, le modèle ne peut que s'approcher de la moyenne : une erreur faible en moyenne peut cacher des cas mal prédits, d'où l'importance du P95 et de l'analyse par régime.
- **Entraînement non convergé.** Les pertes de validation baissaient encore à la 20ᵉ époque : les scores sont un peu sous-estimés, sans que le classement des modèles soit en cause.
- **Une seule graine.** Chaque expérience est entraînée une fois. Les petits écarts entre modèles devraient être confirmés sur plusieurs graines avant d'être tenus pour significatifs.
- **Pas d'incertitude.** Le modèle donne une valeur, sans intervalle de confiance.

**Perspectives**

- **Prédire l'incertitude** (sortie moyenne et variance, perte de vraisemblance gaussienne) : le modèle signalerait lui-même les mesures peu informatives, ce qui est directement utile sur le terrain.
- **Choisir l'excitation** : l'analyse 8.4 donne les caractéristiques des signaux les plus informatifs ; on peut en déduire un protocole de chauffe, comme le fait l'article pour le HLC, mais pour les six indicateurs à la fois.
- **Réduire la durée de mesure** : réentraîner sur les 6, 4 ou 2 premières heures pour tracer la précision en fonction de la durée de l'essai.
- **Recherche d'hyperparamètres** (Optuna) sur le modèle retenu : largeur, profondeur, taux d'apprentissage, taille des patches.